In [5]:
import torch
import math
from torch.distributions import Normal

In [6]:
def brownian_motion(n_paths, n_steps, seed, S0=100.0, sigma=0.2, T=30/252):
    dt = T / n_steps
    g = torch.Generator().manual_seed(seed)
    Z = torch.randn(n_paths, n_steps, generator=g)
    log_returns = -0.5 * sigma**2 * dt + sigma * dt**0.5 * Z
    log_paths = torch.cumsum(log_returns, dim=1)
    log_paths = torch.cat([torch.zeros(n_paths, 1), log_paths], dim=1)
    return S0 * torch.exp(log_paths)

In [7]:
paths = brownian_motion(100_000, 30, seed=0)
print(paths.shape)                          
print(paths[:, 0].unique())                  
print(paths[:, -1].mean())                  
print(torch.log(paths[:, -1] / 100).std()) 

torch.Size([100000, 31])
tensor([100.])
tensor(99.9671)
tensor(0.0690)


In [ ]:
def bs_call_price(S0, K, T, sigma):
    d1 = (math.log(S0 / K) + 0.5 * sigma**2 * T) / (sigma * math.sqrt(T))
    d2 = d1 - sigma * math.sqrt(T)
    N = lambda x: 0.5 * (1 + math.erf(x / math.sqrt(2)))  
    return S0 * N(d1) - K * N(d2)

def bs_delta(paths, K, T, sigma):
    n_steps = paths.shape[1] - 1
    dt = T / n_steps
    tau = T - torch.arange(n_steps) * dt     
    S = paths[:, :-1]                        
    d1 = (torch.log(S / K) + 0.5 * sigma**2 * tau) / (sigma * torch.sqrt(tau))
    return Normal(0.0, 1.0).cdf(d1)

In [10]:
K = 100.0
T = 30/252
sigma = 0.2

premium = bs_call_price(100.0, K, T, sigma)
print(premium)                        

paths = brownian_motion(100_000, 30, seed=0)
delta = bs_delta(paths, K, T, sigma)
print(delta.shape)                    
print(delta[:, 0].unique())           
print(delta.min(), delta.max())        
print(delta[paths[:, -1] > 110, -1].mean())   
print(delta[paths[:, -1] < 90, -1].mean()) 

2.7524171533585005
torch.Size([100000, 30])
tensor([0.5138])
tensor(0.) tensor(1.)
tensor(1.)
tensor(3.9357e-11)


In [11]:
def hedging_pnl(paths, delta, c, K, premium):
   
    S = paths[:, :-1]                           
    dS = paths[:, 1:] - paths[:, :-1]         

    gains = (delta * dS).sum(dim=1)            

    prev = torch.cat([torch.zeros_like(delta[:, :1]), delta[:, :-1]], dim=1)
    trades = (delta - prev).abs()               
    costs = c * (trades * S).sum(dim=1)        

    payoff = torch.clamp(paths[:, -1] - K, min=0)   

    pnl = premium - payoff + gains - costs
    turnover = trades.sum(dim=1)
    return pnl, turnover

In [12]:
for c in [0.0, 0.001, 0.005]:
    pnl, turnover = hedging_pnl(paths, delta, c, K, premium)
    print(f"c={c}: mean PnL {pnl.mean():.4f}, std {pnl.std():.4f}, turnover {turnover.mean():.3f}")

c=0.0: mean PnL 0.0004, std 0.4307, turnover 2.224
c=0.001: mean PnL -0.2219, std 0.4450, turnover 2.224
c=0.005: mean PnL -1.1110, std 0.5677, turnover 2.224
